# 02 — Feature Engineering for Attn-LOB Pretraining

Builds the labeled dataset for the mid-price direction pretraining task (paper Section III-A/III-B), using `paper_replication.features`.

Adaptations from the paper, since our data is a ~10s snapshot grid rather than raw per-event data:
- `window_T` / `horizon_k` stay row-count based (matching the paper's own event-count definition), but at our cadence they span minutes rather than milliseconds.
- The label threshold `alpha` is calibrated from the data instead of reused from the paper (theirs was tuned to a different asset's tick-level noise).
- OSI is proxied from columns already in the LOB snapshot file (`net_trade_sign`, `trade_count`, `ema_ofi`); the faithful version built from raw ticks exists in `dynamic_state.order_strength_index_from_ticks` but is **not** invoked here (69M-row file, expensive to run).
- OSI/RV/RSI only reconstruct the *trade* event category of the paper's definition — new limit orders and cancellations aren't in this dataset at all.

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np

from paper_replication.features import FeatureConfig, build_feature_dataset
from paper_replication.features.dataset import chronological_split

LOB_PATH = "../data/btc_usdt_20221019_20221030_lob.parquet"

## Build the dataset

`use_real_osi` stays `False` (the default) — this only touches the LOB snapshot parquet, not the 69M-row ticks file.

In [ ]:
config = FeatureConfig()
print(config)

dataset = build_feature_dataset(LOB_PATH, config)

print(f"n_samples          = {len(dataset)}")
print(f"lob_state shape     = {dataset.lob_state.shape}")
print(f"dynamic_state shape = {dataset.dynamic_state.shape}")
print(f"dynamic features    = {dataset.dynamic_feature_names}")
print(f"alpha_used          = {dataset.alpha_used:.6g}")

## Class balance

Calibrated `alpha` should put each class near `config.label_target_balance` (1/3 by default).

In [ ]:
labels, counts = np.unique(dataset.labels, return_counts=True)
for lab, cnt in zip(labels, counts):
    name = {-1: "down", 0: "stationary", 1: "up"}[int(lab)]
    print(f"{name:>11}: {cnt:7d}  ({cnt / len(dataset):.1%})")

plt.bar([{-1: "down", 0: "stationary", 1: "up"}[int(l)] for l in labels], counts)
plt.ylabel("count")
plt.title("Label distribution")
plt.show()

## Chronological split + leakage check

No shuffling: train ends strictly before val starts, val strictly before test. Also check each split's own class balance doesn't collapse (e.g. all-one-class in test would mean the split boundary landed in a trend).

In [ ]:
splits = chronological_split(dataset, test_frac=0.5, val_frac_of_train=0.2)

for name, split in splits.items():
    lab, cnt = np.unique(split.labels, return_counts=True)
    balance = {int(l): f"{c / len(split):.1%}" for l, c in zip(lab, cnt)}
    print(f"{name:>5}: n={len(split):6d}  balance={balance}")

assert splits["train"].timestamps[-1] < splits["val"].timestamps[0]
assert splits["val"].timestamps[-1] < splits["test"].timestamps[0]
print("\nNo temporal overlap between splits: OK")

## Normalization sanity check

Spot-check a few random windows: price columns should have ~0 mean / ~1 std within each window, volume columns should max out at ~1.

In [ ]:
price_mask = np.tile([True, False, True, False], config.n_levels)
vol_mask = ~price_mask

rng = np.random.default_rng(0)
sample_idx = rng.choice(len(dataset), size=5, replace=False)

for i in sample_idx:
    window = dataset.lob_state[i]
    price_mean = window[:, price_mask].mean()
    price_std = window[:, price_mask].std()
    vol_max = window[:, vol_mask].max()
    print(
        f"window {i:6d}: price mean={price_mean:+.4f} std={price_std:.4f}  vol max={vol_max:.4f}"
    )

## Dynamic feature sanity plots

RV should track known volatile stretches; OSI should oscillate around 0; RSI (paper's Gain/(Gain+Loss) definition) should stay in [0, 1].

In [ ]:
fig, axes = plt.subplots(
    len(dataset.dynamic_feature_names),
    1,
    figsize=(10, 2 * len(dataset.dynamic_feature_names)),
    sharex=True,
)
for ax, name, col in zip(axes, dataset.dynamic_feature_names, dataset.dynamic_state.T):
    ax.plot(dataset.timestamps, col, linewidth=0.5)
    ax.set_ylabel(name, fontsize=8)
axes[-1].set_xlabel("timestamp")
fig.suptitle("Dynamic state features over time")
fig.tight_layout()
plt.show()

## Visual check: labels against mid-price

Reload `mid_price` for the labeled timestamps (not stored on `AttnLOBDataset`) and confirm up/down labels line up with visible mid-price moves on a short slice.

In [ ]:
import duckdb

mid = duckdb.query(
    f"SELECT timestamp, mid_price FROM '{LOB_PATH}' ORDER BY timestamp"
).df()
mid_by_ts = mid.set_index("timestamp")["mid_price"]

slice_n = 500
ts_slice = dataset.timestamps[:slice_n]
labels_slice = dataset.labels[:slice_n]
mid_slice = mid_by_ts.reindex(ts_slice).to_numpy()

colors = {-1: "tab:red", 0: "tab:gray", 1: "tab:green"}
plt.figure(figsize=(12, 4))
plt.plot(ts_slice, mid_slice, color="black", linewidth=0.7, zorder=1)
for lab, color in colors.items():
    mask = labels_slice == lab
    plt.scatter(
        ts_slice[mask], mid_slice[mask], color=color, s=8, label=str(lab), zorder=2
    )
plt.legend(title="label")
plt.xlabel("timestamp")
plt.ylabel("mid_price")
plt.title(f"First {slice_n} labeled samples: mid-price colored by direction label")
plt.show()

## Save processed dataset

Cache the labeled arrays so the model-building notebook doesn't need to rebuild them from the raw parquet each time.

In [ ]:
import pathlib

out_dir = pathlib.Path("../data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

np.savez(
    out_dir / "attn_lob_pretrain.npz",
    lob_state=dataset.lob_state,
    dynamic_state=dataset.dynamic_state,
    dynamic_feature_names=np.array(dataset.dynamic_feature_names),
    labels=dataset.labels,
    timestamps=dataset.timestamps,
    alpha_used=dataset.alpha_used,
)
print(f"Saved to {out_dir / 'attn_lob_pretrain.npz'}")